# Lab 02.5 – Dual-LED PWM Controller and Timing Investigation

**Student:** ____________________   **Group:** __________   **Date:** __________

Individual assignment following Lab02.1–Lab02.4. Use only the Zybo legacy board, the existing `base.bit` overlay and Python/Jupyter.

- **Task 5 — required:** complementary LED brightness, manual controls and automatic fade (estimated 60–90 min).
- **Task 5+ — optional:** measure the average software PWM frequency and investigate console-output overhead (estimated 30–45 min).

The initialization is provided. Complete the application logic yourself; TODO cells are deliberately unfinished. A `NotImplementedError` marks work to complete, not a hardware fault. Do not simply run all cells expecting a finished application.

## Objectives and limitations

Combine timing, GPIO, PWM and button events; distinguish the PWM period from the fade-update interval; test boundary conditions; compare requested timing with measured execution time.

This is **software PWM under Linux**, not a hardware PWM peripheral. Requested frequencies and duty cycles are approximate. No extra sensors or measurement instruments are required. Perceived brightness need not be linear with duty cycle.

Run only one GPIO-control loop at a time. Stop Task 5 before running Task 5+. On normal exit or interruption, switch off all four LEDs using a `finally` block.

## 1. Hardware initialization

Use the same overlay and GPIO names as the preceding labs. Run this cell before the exercises. If a GPIO name is missing, inspect `ol.ip_dict` and check that the same Lab02 overlay is being used; do not substitute a different bitstream blindly.

In [ ]:
from pynq import Overlay
import time

ol = Overlay("base.bit")

leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)

buttons = ol.btns_gpio.channel1
buttons.setdirection("in")
buttons.setlength(4)

switches = ol.switches_gpio.channel1
switches.setdirection("in")
switches.setlength(4)

leds.write(0, 0b1111)
print("GPIO initialized. Application logic must be completed below.")

# Task 5 — Dual-LED PWM Controller

## 2. Complementary PWM

Generate a **single shared PWM period** for LD0 and LD1:

- LD0 is ON for fraction `D / 100` of the period.
- LD1 is ON for the remaining fraction.
- LD2 and LD3 remain OFF.
- In each nonzero-duration phase, only one of LD0/LD1 is ON. Switch the pattern using one GPIO write.
- Do not generate a multi-second burst on LD0 followed by another burst on LD1.

For requested frequency `f`:

\[
T=1/f,\qquad t_0=T D/100,\qquad t_1=T-t_0.
\]

At **D = 0%**, LD0 stays OFF and LD1 stays ON for the entire nominal period. At **D = 100%**, the reverse applies. Do not briefly write the unwanted pattern at either endpoint, and do not eliminate the period delay at the endpoints.

Your function `dual_pwm_period(duty, frequency)` must generate exactly **one nominal period**, validate that frequency is positive, and constrain duty to 0…100%. It should not print or switch both LEDs off after every period.

## 3. Controls and state

| Input | Manual mode | Automatic mode |
|---|---|---|
| BTN0 | Increase D by 5 percentage points | Ignore |
| BTN1 | Decrease D by 5 percentage points | Ignore |
| BTN2 | Set D to 50% | Ignore |
| BTN3 | Stop and switch off all LEDs | Stop and switch off all LEDs |
| SW0 | 0 = manual | 1 = automatic |
| SW1 | No effect on manual brightness | 0 = slow fade; 1 = fast fade |
| SW3:SW2 | Select requested PWM frequency | Select requested PWM frequency |

| SW3:SW2 | Requested frequency |
|---|---:|
| 00 | 25 Hz |
| 01 | 50 Hz |
| 10 | 100 Hz |
| 11 | 200 Hz |

Start with **D = 50%**; read the current switch configuration on entry.

**Automatic fade**
- Change D by 5 percentage points per update, reverse direction at 0% and 100%, and repeat.
- Slow: one duty update every **0.20 s**, giving a nominal complete 0→100→0 sweep of **8 s**.
- Fast: one update every **0.10 s**, giving a nominal complete sweep of **4 s**.
- Entering automatic mode preserves D, sets the direction upward (downward if D is already 100%), and restarts the fade timer.
- Returning to manual mode preserves the current D.
- Changing fade speed restarts the fade timer. Changing PWM frequency must not redefine the fade interval.

**Button behavior**
- One physical press gives one command; holding a button does not repeat it.
- Apply software debouncing, not only edge detection. A timestamp-based stable-state filter of about 30 ms is one possible approach; do not add a 50 ms blocking debounce sleep to the PWM loop.
- BTN3 has priority over other buttons. In manual mode BTN2 takes priority over BTN0/BTN1; simultaneous BTN0 and BTN1 cancel.
- Read inputs between individual PWM periods. At 25 Hz a nominal period already takes 40 ms, so short taps may be missed. Use deliberate presses of at least about 150 ms during acceptance testing; explain this sampling limitation.

Print status on accepted configuration or manual-duty changes, **not on every PWM period or automatic fade step**.

## 4. Design before coding

Complete this table before implementing the controller.

| Design question | Your answer |
|---|---|
| GPIO pattern for the LD0 phase, with LD2/LD3 OFF | |
| GPIO pattern for the LD1 phase, with LD2/LD3 OFF | |
| T, t0 and t1 at 100 Hz and D = 25% | |
| Endpoint behavior at D = 0% / 100% | |
| Expression to extract SW3:SW2 as an index 0…3 | |
| State variables for button filtering and new-press detection | |
| State variables for automatic fade | |
| Why a fixed number of PWM periods per fade step changes fade speed when f changes | |

Implementation order: one PWM period → finite endpoint tests → manual controls → automatic fade → live switch changes.

In [ ]:
def dual_pwm_period(duty, frequency):
    """Generate one nominal complementary PWM period on LD0 and LD1."""
    # TODO: validate frequency and constrain duty.
    # TODO: calculate the period and the two phase durations.
    # TODO: handle 0% and 100% without an unwanted short pulse.
    # TODO: write the correct two-LED patterns and delay for nonzero phases.
    raise NotImplementedError("Implement dual_pwm_period before continuing.")

## 5. Finite tests of the PWM function

After completing the function, test D = 0, 25, 50, 75 and 100% at 100 Hz for about two seconds each. Repeat selected tests at 25 Hz and 200 Hz. Visible flicker at the lower frequencies is an observation to record, not automatically a programming error.

Use `try/finally` to clear the LEDs. Compare relative brightness, but do not claim that visual inspection measures the actual duty cycle.

In [ ]:
# Replace this guarded test plan with your finite test loop.
raise NotImplementedError("Write and run the finite PWM tests.")

# TODO: for each duty value, repeatedly call dual_pwm_period
# until the selected test duration has elapsed.
# TODO: ensure leds.write(0, 0b1111) runs in a finally block.

## 6. Student controller

Use a time reference such as `time.monotonic()` to schedule fade updates independently of the requested PWM frequency. The short phase sleeps inside one PWM period are permitted; do not add multi-second bursts or a separate sleep for the fade interval.

Recommended loop:
1. Read switches and buttons; filter and decode inputs.
2. Apply button priority and mode-transition rules.
3. Update D if the fade deadline is due.
4. Generate **one** complementary PWM period.
5. Repeat.

A stable-state filter stores a candidate button value and the time at which it last changed, then accepts it only after it has remained stable long enough. New-press detection operates on the accepted state. Document your chosen method.

Keep the application responsive to deliberate button presses. Explain why this architecture cannot guarantee real-time response under Linux.

In [ ]:
FREQUENCIES = (25, 50, 100, 200)
FADE_INTERVALS = (0.20, 0.10)

def run_controller():
    # TODO: initialize D, mode, fade direction and timestamps.
    # TODO: initialize raw/candidate/stable button states.
    # TODO: prevent held startup buttons from becoming unintended commands.
    raise NotImplementedError("Implement the controller, then remove this guard.")

    try:
        while True:
            # TODO: read and decode switches.
            # TODO: debounce buttons and determine new presses.
            # TODO: handle BTN3 first, then other commands as specified.
            # TODO: handle mode transitions and fade updates.
            # TODO: print only the relevant status changes.
            # TODO: call dual_pwm_period once.
            pass
    finally:
        leds.write(0, 0b1111)
        print("Controller stopped; all LEDs are OFF.")

# A function-definition cell does not start the controller.

In [ ]:
# Run only after implementing the function above.
run_controller()

## 7. Acceptance checklist

- [ ] D = 0% and 100% give the specified steady endpoint states.
- [ ] D = 25% and 75% swap the relative LED brightness.
- [ ] LD2 and LD3 remain OFF throughout.
- [ ] D stays within 0…100%; buttons change it by five percentage points.
- [ ] Holding BTN0/BTN1 gives one change only.
- [ ] BTN2, simultaneous presses and BTN3 obey the priority rules.
- [ ] Changing SW3:SW2 changes the requested PWM frequency during execution.
- [ ] SW0 changes mode without an unintended jump in D.
- [ ] Automatic fade reverses at both endpoints.
- [ ] Slow/fast complete sweeps are approximately 8/4 s; measure between consecutive visits to D = 0%.
- [ ] Compare sweep duration at 25 and 200 Hz and explain any deviation.
- [ ] BTN3 and a Jupyter kernel interrupt leave all LEDs OFF.

Record at least one fault you encountered and how you diagnosed it.

**Observed behavior / corrections:**  
_Write your notes here._

# Task 5+ — How Accurate Is Software PWM?

## 8. Measurement protocol

Stop the interactive controller first. Use the same `dual_pwm_period` function without button polling, fade logic or additional loop delays.

For **each** requested frequency 25, 50, 100 and 200 Hz:
1. Use D = 50% and **N = 100 periods**.
2. Measure the entire N-period loop using `time.perf_counter()`.
3. Repeat **three** times. Do not print inside the baseline timed loop.
4. Print results only after the end timestamp.
5. Compute the ideal duration, average effective frequency and signed percentage error:

\[
t_{ideal}=N/f_{requested},\qquad
f_{effective}=N/\Delta t,\qquad
e_f=100\,(f_{effective}-f_{requested})/f_{requested}.
\]

Keep the setup and final LED cleanup outside the timed interval. Include the loop and PWM function calls in the measurement. A run should be recorded only if all N periods completed.

**Console-output experiment:** repeat the 100 Hz test three times, still with N = 100 and D = 50%, but print one short line inside the timed loop after each period. Keep everything else unchanged. Expect more output; do not extend this to an unlimited loop.

In [ ]:
def measure_pwm(frequency, periods=100, verbose=False):
    """Return one completed measurement as a dictionary."""
    # TODO: validate the inputs before starting the timer.
    # TODO: start perf_counter, then execute exactly 'periods' PWM periods.
    # TODO: in verbose mode, print one line after each period INSIDE the timing.
    # TODO: stop the timer BEFORE cleanup or summary printing.
    # TODO: use finally to switch off all LEDs, including on interruption.
    # TODO: calculate and return the fields for the results table.
    raise NotImplementedError("Implement the measurement function.")

# Store results in a list of dictionaries; no additional package is required.
results = []
# TODO: 3 baseline repetitions at each frequency.
# TODO: 3 verbose repetitions at 100 Hz.
# TODO: print a formatted results table after each completed run or at the end.

## 9. Results and interpretation

Replace the empty rows with your measurements. Include all 12 baseline runs and all 3 verbose runs; the rows below illustrate the required fields.

| Mode | Requested f (Hz) | Run | N | Ideal time (s) | Measured time (s) | Effective f (Hz) | Signed error (%) |
|---|---:|---:|---:|---:|---:|---:|---:|
| Baseline | 25 | 1 | 100 | | | | |
| Baseline | 50 | 1 | 100 | | | | |
| Baseline | 100 | 1 | 100 | | | | |
| Baseline | 200 | 1 | 100 | | | | |
| Verbose | 100 | 1 | 100 | | | | |

For each configuration, report the mean and min–max range of the three effective-frequency estimates. Compare the mean baseline and verbose values at 100 Hz.

**What this experiment measures:** average software-loop throughput, used as an estimate of the average generated PWM frequency. Timing includes Python, GPIO calls and sleep overhead; the verbose test also includes the Python/Jupyter output path.

**What it does not measure:** individual pulse widths, actual duty-cycle error, electrical transition timing or cycle-to-cycle jitter. Differences between repeated runs are not a direct measurement of PWM jitter. Jupyter may buffer output, so the measured overhead is not necessarily the time until the browser finishes rendering every line.

Do not invent measurements. If output overhead is small or inconsistent, report that result and discuss buffering and operating-system scheduling.

## 10. Questions and submission

Answer briefly, using your observations:
1. Why must LD0 and LD1 share the same PWM period rather than separate long bursts?
2. Why must the 0% and 100% cases still consume one nominal period?
3. Why is edge detection alone not equivalent to debouncing?
4. Why should fade timing use elapsed time rather than a fixed number of PWM periods?
5. Why may very short button taps be missed, particularly at 25 Hz?
6. Why can the measured effective frequency differ from the requested frequency?
7. What does the console-output experiment show on your board?
8. Which measurements would require an oscilloscope or logic analyzer?
9. Which part would you move into programmable logic for accurate waveform timing, and what would Python continue to control?

Submit this notebook with your name, completed design table, working Task 5 code, acceptance notes and brief answers. If you complete Task 5+, include its measurement code, all measured results and conclusions. Retain useful output, but clear repetitive verbose lines before submission.

**Final conclusions:**  
_Write your conclusions here._